In [1]:
import keras_tuner
from tensorflow import keras
import sklearn.model_selection as sm
import numpy as np
from keras import losses

2022-09-23 02:14:55.990609: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2022-09-23 02:14:55.990648: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.


In [18]:
kmodes=[0,1,2,3]
par=np.loadtxt('data/params_LH.txt')
mh=np.mean(par[:,0])

ms=np.std(par[:,0])

t=np.empty([len(par),2])
i=0
for el in par:
    t[i,:]=((el-[mh,0])/[ms,1])/4
    i+=1
print(np.shape(t))
fn=open('data/params_t','w+')
for row in t:
    np.savetxt(fn,[row])
fn.truncate()
fn.close()

(2167, 2)


In [17]:
print(np.max(t[:,0]),np.min(t[:,0]))

0.4328129267019358 -0.4328129267019357


In [19]:
def build_model(hp):
    model=keras.Sequential()
    model.add(keras.layers.Dense(2,activation='relu'))
    for i in range(hp.Int("num_layers",1,32)):
        model.add(
            keras.layers.Dense(
                #tune number of neurons
                units=hp.Int("units",min_value=16,max_value=512,step=4),
                #tune activation function to use
                activation=hp.Choice("activation",['relu','elu','softmax','exponential','linear'])
                    #,'softmin','sigmoid','softplus','softsign','selu']
                )
            )
    #Tune whether to use dropout
    dropout_rate=hp.Float("dropout rate",min_value=0.005,max_value=0.5)
    if hp.Boolean("dropout"):
        model.add(keras.layers.Dropout(rate=dropout_rate))
    model.add(keras.layers.Dense(4,activation='relu'))
    #Defining optimizer learning rate as a hyperparameters
    learning_rate=hp.Float("lr",min_value=1e-8,max_value=1e-4,sampling='log')
    model.compile(
                optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
                loss="mse",
                metrics=["accuracy"]
                )
    return model

In [30]:
build_model(keras_tuner.HyperParameters())
mod_name="cii"
# kmode=input("Enter k mode axis: ")
kmode=[0]
tuner=keras_tuner.Hyperband(
    hypermodel=build_model,
    objective='accuracy',
    max_epochs=10,
    factor=3,
    hyperband_iterations=5,
    # seed=None,
    # hyperparameters=None,
    tune_new_entries=True,
    allow_new_entries=True,
    # **kwargs
)
# tuner = keras_tuner.BayesianOptimization(
#             hypermodel=build_model,
#             objective='accuracy',
#             max_trials=500,
#             executions_per_trial=3,
#             directory='tuning_results',
#             project_name=mod_name
#             )

In [21]:
import math as m
pi=m.pi
k=np.loadtxt('data/k.txt')
k=k[[0,1,2,3]]
# npk=np.loadtxt('data/Npk.txt',usecols=(2,3,4,5))
# fn=open('data/cii_dpk','w+')
# for el in npk:
#     dpk=((k**3)*el)/(2*pi**2)
#     np.savetxt(fn,[dpk])
# fn.truncate()
# fn.close()

In [22]:
dpk=np.loadtxt('data/cii_dpk')
n=np.loadtxt('data/nbins.txt')
n=n[[0,1,2,3]]
print(np.shape(k),np.shape(n),np.shape(dpk))

(4,) (4,) (2167, 4)


In [24]:
print(np.min(dpk),np.max(dpk))

30.794477092592867 224650.27296421953


In [25]:
path = 'data/'
params = np.loadtxt(path+'params_t')#params

pk=(np.log(dpk)/15)
print(np.min(pk),np.max(pk))

0.2284890239150056 0.8214866753041318


In [26]:
print(np.max(params[:,0])-np.min(params[:,0]))

0.8656258534038714


In [27]:
params_traino,params_test,pk_traino,pk_test = sm.train_test_split(params,pk, test_size=0.1,shuffle=False)
params_train,params_val,pk_train,pk_val = sm.train_test_split(params_traino,pk_traino, test_size=0.2,shuffle=False)

# scaler_1=MinMaxScaler(feature_range=(0, 1))
# scaler_1.fit(params_train)
# params_train=scaler_1.transform(params_train)
# scaler_4=MinMaxScaler(feature_range=(0, 1))
# scaler_4.fit(params_test)
# params_test=scaler_4.transform(params_test)

In [29]:
print(np.min(pk)-np.max(pk))

-0.5929976513891262


In [31]:
# warnings.simplefilter('ignore')
# warnings.filterwarnings('ignore')
tuner.search(params_train, pk_train, epochs=20, validation_data=(params_val, pk_val))

Trial 9 Complete [00h 00m 02s]
accuracy: 0.0

Best accuracy So Far: 1.0
Total elapsed time: 00h 00m 39s

Search: Running Trial #10

Value             |Best Value So Far |Hyperparameter
31                |11                |num_layers
468               |476               |units
linear            |softmax           |activation
0.078597          |0.10094           |dropout rate
True              |False             |dropout
1.9895e-08        |1.8011e-05        |lr
2                 |2                 |tuner/epochs
0                 |0                 |tuner/initial_epoch
2                 |2                 |tuner/bracket
0                 |0                 |tuner/round

Epoch 1/2
26/49 [==============>...............] - ETA: 1s - loss: 0.2643 - accuracy: 0.0000e+00

KeyboardInterrupt: 

In [32]:
tuner.results_summary()

Results summary
Results in ./untitled_project
Showing 10 best trials
Trial summary
Hyperparameters:
num_layers: 11
units: 476
activation: softmax
dropout rate: 0.10094281093161799
dropout: False
lr: 1.8011304513421146e-05
tuner/epochs: 2
tuner/initial_epoch: 0
tuner/bracket: 2
tuner/round: 0
Score: 1.0
Trial summary
Hyperparameters:
num_layers: 13
units: 196
activation: elu
dropout rate: 0.18932806383321613
dropout: True
lr: 1.066042196298793e-07
tuner/epochs: 2
tuner/initial_epoch: 0
tuner/bracket: 2
tuner/round: 0
Score: 0.949999988079071
Trial summary
Hyperparameters:
num_layers: 17
units: 492
activation: linear
dropout rate: 0.4302740090351985
dropout: False
lr: 1.726646680539652e-08
tuner/epochs: 2
tuner/initial_epoch: 0
tuner/bracket: 2
tuner/round: 0
Score: 0.5365384817123413
Trial summary
Hyperparameters:
num_layers: 13
units: 340
activation: softmax
dropout rate: 0.35798258882180545
dropout: True
lr: 2.0535678009877794e-05
tuner/epochs: 2
tuner/initial_epoch: 0
tuner/bracket: 